In [1]:
import os

if os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/mimic-sepsis-ews'
else:
    project_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

processed_path = f'{project_path}/data/processed'
print(f"Environment: {'Colab' if 'drive' in project_path else 'Local VSCode'}")
print(f"Project path: {project_path}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

for root, dirs, files in os.walk(project_path):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(project_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

df = pd.read_csv(f'{processed_path}/master_features.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nSepsis label distribution:")
print(df['sepsis_label'].value_counts())
print(f"\nSepsis rate: {df['sepsis_label'].mean():.1%}")
print(f"\nColumns:\n{df.columns.tolist()}")

Environment: Local VSCode
Project path: d:\mimic-sepsis-ews
mimic-sepsis-ews/
  .gitignore
  LICENSE
  README.md
  data/
    processed/
      .gitkeep
      master_features.csv
    raw/
      .gitkeep
  notebooks/
    01_cohort_definition.ipynb
    02_feature_engineering.ipynb
    03_model_training.ipynb
    04_fhir_output.ipynb
  src/
    cohort.py
    contamination_audit.py
    fhir_generator.py
    model.py
Dataset shape: (29405, 41)

Sepsis label distribution:
sepsis_label
0    25014
1     4391
Name: count, dtype: int64

Sepsis rate: 14.9%

Columns:
['stay_id', 'hadm_id', 'subject_id', 'sepsis_label', 'admission_age', 'gender', 'insurance', 'race', 'charlson_comorbidity_index', 'avg_hr', 'max_hr', 'min_hr', 'avg_sbp', 'min_sbp', 'avg_dbp', 'avg_mbp', 'min_mbp', 'avg_rr', 'max_rr', 'avg_temp', 'max_temp', 'min_temp', 'avg_spo2', 'min_spo2', 'vital_measurement_count', 'avg_wbc', 'max_wbc', 'min_wbc', 'avg_lactate', 'max_lactate', 'avg_ph', 'min_ph', 'avg_pao2fio2', 'avg_creatinine', 

In [2]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f"Features with missing values: {len(missing_df)}")
print(f"\n{missing_df.to_string()}")

Features with missing values: 29

                 missing_count  missing_pct
avg_pao2fio2             25998         88.4
avg_glucose              24057         81.8
max_lactate              23994         81.6
avg_lactate              23994         81.6
max_creatinine           23950         81.4
avg_bun                  23924         81.4
avg_creatinine           23950         81.4
min_bicarbonate          23892         81.3
avg_bicarbonate          23892         81.3
avg_sodium               23631         80.4
avg_potassium            23614         80.3
avg_wbc                  22526         76.6
max_wbc                  22526         76.6
min_wbc                  22526         76.6
min_ph                   22439         76.3
avg_ph                   22439         76.3
avg_temp                  1231          4.2
min_temp                  1231          4.2
max_temp                  1231          4.2
insurance                  459          1.6
avg_rr                     185          0.

In [3]:
from sklearn.impute import SimpleImputer

# Separate IDs, label, and categorical features
id_cols = ['stay_id', 'hadm_id', 'subject_id']
label_col = 'sepsis_label'
cat_cols = ['gender', 'insurance', 'race']

# Numeric feature columns only
num_cols = [c for c in df.columns
            if c not in id_cols + [label_col] + cat_cols]

# Step 1: Create missingness indicator flags for high-missing lab features
high_missing = ['avg_pao2fio2', 'avg_lactate', 'max_lactate',
                'avg_ph', 'min_ph']

for col in high_missing:
    df[f'{col}_missing'] = df[col].isnull().astype(int)
    print(f"Created flag: {col}_missing")

# Step 2: Median imputation for all numeric features
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])

# Step 3: Mode imputation for categorical
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Step 4: Encode categoricals
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"\nAfter imputation - missing values: {df.isnull().sum().sum()}")
print(f"Dataset shape after encoding: {df.shape}")

Created flag: avg_pao2fio2_missing
Created flag: avg_lactate_missing
Created flag: max_lactate_missing
Created flag: avg_ph_missing
Created flag: min_ph_missing

After imputation - missing values: 0
Dataset shape after encoding: (29405, 80)


In [4]:
from sklearn.model_selection import train_test_split

# Define feature matrix and label
id_cols = ['stay_id', 'hadm_id', 'subject_id']
label_col = 'sepsis_label'

X = df.drop(columns=id_cols + [label_col])
y = df[label_col]

# 80/20 stratified split to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nTraining sepsis rate: {y_train.mean():.1%}")
print(f"Test sepsis rate:     {y_test.mean():.1%}")
print(f"\nFeatures: {X_train.shape[1]}")

Training set: (23524, 76)
Test set:     (5881, 76)

Training sepsis rate: 14.9%
Test sepsis rate:     14.9%

Features: 76


In [5]:
# Save processed splits to Drive for use in other notebooks
import numpy as np

processed_path = f'{project_path}/data/processed'

X_train.to_csv(f'{processed_path}/X_train.csv', index=False)
X_test.to_csv(f'{processed_path}/X_test.csv', index=False)
y_train.to_csv(f'{processed_path}/y_train.csv', index=False)
y_test.to_csv(f'{processed_path}/y_test.csv', index=False)

print("Saved:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

Saved:
  X_train: (23524, 76)
  X_test:  (5881, 76)
  y_train: (23524,)
  y_test:  (5881,)
